# 03 · Stars, galaxies, and a roll

[Notebook 02](02_disperse_a_star.ipynb) dispersed a point source. Real fields also contain **galaxies**, which are *extended* — their light has a shape on the sky that gets sheared and smeared by the dispersion. This notebook adds the galaxy path, builds a small **mixed field** of stars and galaxies, and then re-observes that field at a different **roll angle** to see the layout change on the detector.

You will: generate galaxy morphologies with `sersic`, disperse an extended source with `galaxy_disperser`, co-add a handful of stars and galaxies into one SCA image, and rotate the whole field to mimic an observatory roll. The roll sets up the **contamination** question we tackle in notebook 06: which spectra land on top of which.

## 0 · Setup

In [ ]:
import os
from pathlib import Path
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR",
                      str(Path.home() / ".cache" / "roman_grs_jax"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm
import jax
import jax.numpy as jnp

from roman_disperser import (paths, psf_model, star_disperser,
                             galaxy_disperser, sersic)
from roman_disperser.elements import GRISM
from roman_disperser.optical_model import RomanOpticalModel
import roman_disperser.optical_model_jax as omj
import tutorial_helpers as th

SCA = 5
DETECTOR = f"WFI{SCA:02d}"
element = GRISM
model = RomanOpticalModel(config_file=str(paths.optical_model_path(element=element)))

# Order-1 payloads, and BOTH dispersers off the same optics + PSF. The element
# keeps the PSF's STPSF filter and wavelength grid consistent with the band.
opt = omj.make_sca_payload(model, sca=SCA, order="1")
psf = psf_model.get_or_make_psf_payload(detector=DETECTOR, order="1", element=element,
                                        cache_dir=str(paths.psf_cache_dir()), verbose=False)
star_disp = star_disperser.make_star_disperser(psf, opt)
gal_disp = galaxy_disperser.make_galaxy_disperser(psf, opt)

OVERSAMPLE = psf["oversample"]          # 4; galaxy images are built at this resolution
GAL_NPIX = 30 * OVERSAMPLE              # fixed stamp size (keeps the JIT shape constant)
wl_um, _, _ = th.wavelength_grid(element)         # 2 Å production grid over the element's band
wl_j = jnp.asarray(wl_um)
print("JAX backend:", jax.default_backend(), "| oversample:", OVERSAMPLE,
      "| galaxy stamp:", GAL_NPIX)

## 1 · Galaxy morphology — Sérsic profiles

We model galaxy morphology with a **Sérsic profile** via `sersic.make_sersic_image(r_eff_pix, n, ba, theta, npix)` (index `n`, axis ratio `ba`, orientation `theta`). `sersic.catalog_r_eff_to_pixels` converts a half-light radius in arcsec to **oversampled** pixels — we build stamps at the PSF oversampling so the disperser can convolve them directly.

In [ ]:
reff_pix = sersic.catalog_r_eff_to_pixels(0.4, pixel_scale=0.11, oversample=OVERSAMPLE)
print(f"r_eff = 0.4 arcsec -> {reff_pix:.1f} oversampled pixels")

examples = [("n=1, round",       dict(n=1.0, ba=0.9, theta=0.0)),
            ("n=1, inclined",    dict(n=1.0, ba=0.4, theta=np.deg2rad(30))),
            ("n=4, bulge",       dict(n=4.0, ba=0.7, theta=np.deg2rad(70)))]

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, (label, kw) in zip(axes, examples):
    img = np.asarray(sersic.make_sersic_image(reff_pix, npix=GAL_NPIX, **kw))
    ax.imshow(img, origin="lower", cmap="magma",
              norm=AsinhNorm(linear_width=img.max() * 0.02))
    ax.set(title=label, xticks=[], yticks=[])
fig.suptitle("Sérsic galaxy stamps (oversampled)")
fig.tight_layout()

## 2 · Dispersing an extended source

`galaxy_disperser.make_galaxy_disperser(psf, opt)` returns a callable

```python
disperse(image, x0, y0, spectrum, wavelengths_um, output) -> output
```

where `image` is the oversampled galaxy stamp (normalised to unit sum — the flux is carried by `spectrum`), and `x0, y0` is the **undispersed centre** in 1-indexed SCA pixels. Note the argument order differs from the star disperser, which takes `(x, y, wavelengths, counts, output)`.

Below we disperse a star and a galaxy at the *same* position with the *same* brightness. The galaxy's spectrum is smeared not just along the dispersion direction but **across** it too — the source's spatial extent is convolved into the spectral trace, broadening it. (Internally the disperser warps the stamp through the dispersion Jacobian; that machinery is the subject of notebook 07.)

In [ ]:
x0, y0 = 2000.0, 2000.0
zeros = lambda: jnp.zeros((4088, 4088), jnp.float32)

# star
_, c_star = th.template_to_counts("g0v", 18.0, sca=SCA, order="1", wl_um=wl_um)
star_img = np.asarray(star_disp(x0, y0, wl_j, jnp.asarray(c_star), zeros()))

# galaxy at the same place, same brightness, redshifted into the band
g = sersic.make_sersic_image(reff_pix, n=1.0, ba=0.5, theta=np.deg2rad(30), npix=GAL_NPIX)
g = jnp.asarray(g / g.sum())
_, c_gal = th.template_to_counts("kc96_starb1", 18.0, sca=SCA, order="1",
                                 wl_um=wl_um, redshift=1.2)
gal_img = np.asarray(gal_disp(g, x0, y0, jnp.asarray(c_gal), wl_j, zeros()))

def bbox(img, pad=12):
    ys, xs = np.nonzero(img)
    return xs.min()-pad, xs.max()+pad, ys.min()-pad, ys.max()+pad

fig, (a0, a1) = plt.subplots(1, 2, figsize=(9, 5))
for ax, img, title in [(a0, star_img, "point source (star)"),
                       (a1, gal_img, "extended source (galaxy)")]:
    bx0, bx1, by0, by1 = bbox(img)
    ax.imshow(img[by0:by1, bx0:bx1], origin="lower", cmap="inferno",
              norm=AsinhNorm(linear_width=img.max()*0.02, vmin=0, vmax=img.max()))
    ax.set(title=title, xticks=[], yticks=[])
fig.tight_layout()

## 3 · A small mixed field

Now a small field — several stars and galaxies at random positions — co-added into one image, exactly as the pipeline does for a real pointing (just much smaller). Each source contributes through its own count-rate spectrum.

In [ ]:
rng = np.random.default_rng(42)
CX = CY = 2044.0          # SCA centre (detector is 4088 x 4088)
R_MAX = 1300.0            # keep sources within this radius so a roll keeps them on-chip

def random_positions(n):
    r = R_MAX * np.sqrt(rng.uniform(0.02, 1.0, n))     # uniform over the disc
    a = rng.uniform(0, 2*np.pi, n)
    return CX + r*np.cos(a), CY + r*np.sin(a)

# stars: (x, y, mag)
sx, sy = random_positions(9)
stars = list(zip(sx, sy, rng.uniform(17.5, 20.0, 9)))

# galaxies: (x, y, mag, z, n, ba, theta_deg, r_eff_arcsec, template)
gx, gy = random_positions(5)
TEMPLATES = ["kc96_starb1", "kc96_elliptical"]
galaxies = [(x, y, float(rng.uniform(20.0, 21.5)), float(rng.uniform(1.0, 2.0)),
             float(rng.choice([1.0, 1.5, 4.0])), float(rng.uniform(0.3, 0.9)),
             float(rng.uniform(0, 180)), float(rng.uniform(0.2, 0.5)),
             TEMPLATES[int(rng.integers(2))])
            for x, y in zip(gx, gy)]
print(f"{len(stars)} stars + {len(galaxies)} galaxies")

def roll_xy(x, y, roll_deg):
    # rotate about the SCA centre (our stand-in for an observatory roll)
    t = np.deg2rad(roll_deg); c, s = np.cos(t), np.sin(t)
    return CX + c*(x-CX) - s*(y-CY), CY + s*(x-CX) + c*(y-CY)

def disperse_field(roll_deg):
    out = jnp.zeros((4088, 4088), jnp.float32)
    for x, y, mag in stars:
        xr, yr = roll_xy(x, y, roll_deg)
        _, c = th.template_to_counts("g0v", mag, sca=SCA, order="1", wl_um=wl_um)
        out = star_disp(float(xr), float(yr), wl_j, jnp.asarray(c), out)
    for x, y, mag, z, n, ba, th0, reff, tmpl in galaxies:
        xr, yr = roll_xy(x, y, roll_deg)
        rpix = sersic.catalog_r_eff_to_pixels(reff, 0.11, oversample=OVERSAMPLE)
        gi = sersic.make_sersic_image(rpix, n, ba, np.deg2rad(th0 + roll_deg), GAL_NPIX)
        gi = jnp.asarray(gi / gi.sum())
        _, c = th.template_to_counts(tmpl, mag, sca=SCA, order="1", wl_um=wl_um, redshift=z)
        out = gal_disp(gi, float(xr), float(yr), jnp.asarray(c), wl_j, out)
    out.block_until_ready()
    return np.asarray(out)

# source positions, to overlay as markers so the roll is obvious
star_xy = np.array([(x, y) for x, y, *_ in stars])
gal_xy  = np.array([(x, y) for x, y, *_ in galaxies])

def mark_sources(ax, roll_deg):
    sx, sy = roll_xy(star_xy[:, 0], star_xy[:, 1], roll_deg)
    gx, gy = roll_xy(gal_xy[:, 0], gal_xy[:, 1], roll_deg)
    ax.plot(sx, sy, "o", mfc="none", mec="cyan", ms=7, mew=1.3, label="star")
    ax.plot(gx, gy, "s", mfc="none", mec="yellow", ms=8, mew=1.3, label="galaxy")

field0 = disperse_field(0.0)
print(f"mixed field: total {field0.sum():.0f} e-/s")

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.imshow(field0, origin="lower", cmap="inferno",
          norm=AsinhNorm(linear_width=field0.max()*0.002, vmin=0, vmax=field0.max()))
mark_sources(ax, 0.0)
ax.legend(loc="upper right", fontsize=8)
ax.set(title=f"mixed field — {len(stars)} stars + {len(galaxies)} galaxies (roll = 0°)",
       xlabel="x [pix]", ylabel="y [pix]")
fig.tight_layout()

## 4 · Rolling the field

The observatory can observe the same patch of sky at different **roll** (position) angles. A real roll rotates the field about the **WFI field centre (WFICEN)**, so on a single SCA — which sits off-axis — sources both rotate *and* translate. To keep the demo on one detector and isolate the effect, we rotate the field about the **SCA centre** instead (the `roll_xy` helper above); this captures the essential change — the spectra reconfigure relative to one another — without the bookkeeping of a full sky pointing. Notebook 07 does the real sky→detector mapping (`omj.get_fpa_pos`).

The dispersion direction is fixed to the detector, so as the sources rotate, *which* spectra fall near *which* sources changes — the seed of the contamination story in notebook 06.

In [ ]:
ROLL = 25.0
field_r = disperse_field(ROLL)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
vmax = max(field0.max(), field_r.max())
for ax, img, deg in [(axes[0], field0, 0.0), (axes[1], field_r, ROLL)]:
    ax.imshow(img, origin="lower", cmap="inferno",
              norm=AsinhNorm(linear_width=vmax*0.002, vmin=0, vmax=vmax))
    mark_sources(ax, deg)
    ax.set(title=f"roll = {deg:.0f}°", xlabel="x [pix]")
axes[0].set_ylabel("y [pix]")
axes[1].legend(loc="upper right", fontsize=8)
fig.suptitle("Same field at two rolls — markers track each source as the field rotates", y=1.0)
fig.tight_layout()

print(f"flux conserved across the roll: {field0.sum():.0f} vs {field_r.sum():.0f} e-/s")

## Recap

- Galaxies are dispersed with `galaxy_disperser.make_galaxy_disperser`, fed an oversampled `sersic` stamp normalised to unit sum; the source's spatial extent broadens the trace across the dispersion direction.
- Mind the argument order: galaxy `(image, x0, y0, spectrum, wavelengths, output)` vs star `(x, y, wavelengths, counts, output)`.
- A field is just many sources co-added into one `(4088, 4088)` image.
- **Roll** rotates the field on the detector (here about the SCA centre as a clean stand-in for a roll about WFICEN); the dispersion axis stays fixed, so the spectral layout reconfigures.

**Next — [04 · Simple spectral extraction](04_simple_extraction.ipynb).** We pull a 1D spectrum back out of a dispersed image — building a minimal trace-based extractor from the optical model.